# Tratamento de Dados
Este notebook aplica as mesmas transformações do `tratativa.ipynb`, mas não altera o arquivo original.
Todas as operações gravam em uma cópia do CSV original.

In [ ]:
import pandas as pd
import re
from pathlib import Path

# Diretório base e utilitário para dados
BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / 'Banco_de_Dados_PII3_AWS').exists() and (BASE_DIR.parent / 'Banco_de_Dados_PII3_AWS').exists():
    BASE_DIR = BASE_DIR.parent
DATA_DIR = BASE_DIR / 'Banco_de_Dados_PII3_AWS'
if not DATA_DIR.exists():
    raise FileNotFoundError(f'“Diretório de dados não encontrado: {DATA_DIR}”')

def get_data_path(nome_arquivo):
    return DATA_DIR / nome_arquivo

def get_copia_path(caminho_arquivo, sufixo='_tratado'):
    caminho = Path(caminho_arquivo)
    if caminho.suffix:
        return caminho.with_name(caminho.stem + sufixo + caminho.suffix)
    return caminho.with_name(caminho.name + sufixo)

In [ ]:
def limpar_avaliacoes(texto):
    if not isinstance(texto, str):
        return texto

    texto = texto.strip()
    texto = re.sub(r'^[_]+', '', texto, flags=re.UNICODE)
    texto = re.sub(r'{2,}', ' ', texto)

    texto = re.sub(r'+([.,;:!?])', r'', texto)
    texto = re.sub(r'([.,;:!?])+', r'', texto)
    texto = re.sub(r'([.,;:!?])(?=[^])', r' ', texto)

    mapeamento = {
        r'\b(n[ãa]o|nã|não)\b': 'não',
        r'\bvc\b': 'você',
        r'\bvcs\b': 'vocês',
        r'\b(mt|mto)\b': 'muito'
    }

    def aplicar_mapeamento(match):
        original = match.group(0)
        alvo = ''
        for padrao, subst in mapeamento.items():
            if re.search(padrao, original, flags=re.IGNORECASE):
                alvo = subst
                break

        if original.isupper():
            return alvo.upper()
        if original and original[0].isupper():
            return alvo.capitalize()
        return alvo

    regex_completo = '|'.join(mapeamento.keys())
    texto = re.sub(regex_completo, aplicar_mapeamento, texto, flags=re.IGNORECASE)

    texto = re.sub(r'(+)([a-z])', lambda m: m.group(1) + m.group(2).upper(), texto)
    texto = re.sub(r'[]+$', '', texto)
    return texto

In [ ]:
def juntar_titulo_mensagem(df, coluna_titulo='review_comment_title', coluna_mensagem='review_comment_message'):
    if coluna_titulo not in df.columns or coluna_mensagem not in df.columns:
        print(f'Erro: colunas {coluna_titulo} ou {coluna_mensagem} não encontradas.')
        return df

    df[coluna_titulo] = df[coluna_titulo].astype('string')
    df[coluna_mensagem] = df[coluna_mensagem].astype('string')

    def combinar(titulo, mensagem):
        titulo = str(titulo).strip() if pd.notna(titulo) else ''
        mensagem = str(mensagem).strip() if pd.notna(mensagem) else ''

        if titulo and mensagem:
            return f'{titulo} - {mensagem}'
        return titulo or mensagem

    df[coluna_mensagem] = df.apply(lambda row: combinar(row[coluna_titulo], row[coluna_mensagem]), axis=1)
    return df

In [ ]:
def processar_csv(caminho_arquivo, nome_coluna, caminho_saida=None):
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo não encontrado em {caminho}')
        return

    if caminho_saida is None:
        caminho_saida = get_copia_path(caminho)

    df = pd.read_csv(caminho)
    if nome_coluna in df.columns:
        print(f'Limpando espacos e padronizando a coluna {nome_coluna}...')
        df[nome_coluna] = df[nome_coluna].apply(limpar_avaliacoes)
        df.to_csv(caminho_saida, index=False)
        print(f'Sucesso! Arquivo gravado em {caminho_saida}')
        return df
    else:
        print(f'Erro: A coluna {nome_coluna} não existe no arquivo CSV.')

In [ ]:
def remover_linhas_sem_review(caminho_arquivo, coluna='review_comment_message', caminho_saida=None):
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo não encontrado em {caminho}')
        return

    if caminho_saida is None:
        caminho_saida = get_copia_path(caminho)

    df = pd.read_csv(caminho)
    if coluna not in df.columns:
        print(f'Erro: A coluna {coluna} não existe no arquivo CSV.')
        return

    antes = len(df)
    df[coluna] = df[coluna].astype('string')
    df = df[df[coluna].str.strip() != '']
    depois = len(df)
    removidas = antes - depois

    df.to_csv(caminho_saida, index=False)
    print(f'Removidas {removidas} linhas sem valor em {coluna}. Arquivo gravado em {caminho_saida}')
    return df

In [ ]:
def apagar_coluna(caminho_arquivo, nome_coluna, caminho_saida=None):
    caminho = Path(caminho_arquivo)
    if not caminho.exists():
        print(f'Erro: Arquivo não encontrado em {caminho}')
        return

    if caminho_saida is None:
        caminho_saida = get_copia_path(caminho)

    df = pd.read_csv(caminho)
    if nome_coluna not in df.columns:
        print(f'Erro: A coluna {nome_coluna} não existe no arquivo CSV.')
        return

    df = df.drop(columns=[nome_coluna])
    df.to_csv(caminho_saida, index=False)
    print(f'Coluna {nome_coluna} apagada com sucesso. Arquivo gravado em {caminho_saida}')
    return df

In [ ]:
def converter_para_string(df, nome_coluna):
    if nome_coluna in df.columns:
        df[nome_coluna] = df[nome_coluna].astype('string')
        print(f
)
    else:
        print(f
    return df

def converter_para_datetime(df, nome_coluna):
    if nome_coluna in df.columns:
        df[nome_coluna] = pd.to_datetime(df[nome_coluna], errors='coerce')
        print(f
)
    else:
        print(f
    return df

In [ ]:
# Exemplo de uso em cópia de arquivo:
original = get_data_path('avaliacoes.csv')
saida = get_copia_path(original, sufixo='_tratada')

df = pd.read_csv(original)
df['review_comment_title'] = df['review_comment_title'].apply(limpar_avaliacoes)
df['review_comment_message'] = df['review_comment_message'].apply(limpar_avaliacoes)
df = juntar_titulo_mensagem(df)
df.to_csv(saida, index=False)
print(f'Arquivo original mantido em {original}')
print(f'Arquivo tratado gravado em {saida}')